<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/09.%20Text%20Processing%20and%20Multiclass%20Classification/ch09_text_multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 9: Text Processing and Multiclass Classification

## Representasi Teks untuk Machine Learning

Machine learning membutuhkan input numerik, namun teks adalah data yang tidak terstruktur. Kita perlu mengubah teks menjadi representasi numerik.

### Bag of Words (BoW)

BoW merepresentasikan dokumen sebagai **frekuensi kata** tanpa mempertimbangkan urutan kata:

$$\text{doc} \rightarrow \text{vector} \in \mathbb{R}^{|V|}$$

di mana $|V|$ adalah ukuran kosakata.

### TF-IDF (Term Frequency - Inverse Document Frequency)

TF-IDF memberikan bobot lebih tinggi pada kata yang sering muncul di dokumen tertentu namun jarang di seluruh corpus:

$$\text{tf}(t, d) = \frac{\text{count}(t \text{ in } d)}{\text{total words in } d}$$

$$\text{idf}(t, D) = \log\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|} + 1$$

$$\text{tfidf}(t, d, D) = \text{tf}(t, d) \times \text{idf}(t, D)$$

Kata seperti "the", "is", "a" (stop words) mendapat skor idf rendah karena muncul di hampir semua dokumen.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Use a subset of 20 newsgroups
categories = ['sci.space', 'rec.sport.hockey', 'comp.graphics',
              'talk.politics.guns', 'rec.motorcycles']

train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)
test_data = fetch_20newsgroups(subset='test', categories=categories,
                                remove=('headers', 'footers', 'quotes'),
                                random_state=42)

print(f'Training samples: {len(train_data.data)}')
print(f'Test samples:     {len(test_data.data)}')
print(f'Categories:       {train_data.target_names}')
print(f'\nSample text (first 200 chars):')
print(train_data.data[0][:200])


## 9.1 CountVectorizer: Bag of Words

CountVectorizer mengubah koleksi dokumen teks menjadi matriks hitungan token (kata).

Parameter penting:
- `max_features`: batasi kosakata ke N kata paling sering
- `min_df`: abaikan kata yang muncul di kurang dari N dokumen
- `max_df`: abaikan kata yang muncul di lebih dari X% dokumen (stop words otomatis)
- `ngram_range`: (1,1) = unigram, (1,2) = unigram+bigram


In [ ]:
# CountVectorizer demonstration
cv = CountVectorizer(max_features=20000, min_df=2, stop_words='english')
X_train_bow = cv.fit_transform(train_data.data)
X_test_bow = cv.transform(test_data.data)

print(f'Vocabulary size: {len(cv.vocabulary_)}')
print(f'Train matrix shape: {X_train_bow.shape} (sparse)')
print(f'Sparsity: {1 - X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1]):.1%}')

# Top words per category
print('\nTop 10 words for each category:')
for cat_idx, cat_name in enumerate(train_data.target_names):
    cat_mask = train_data.target == cat_idx
    cat_text = [train_data.data[i] for i in range(len(train_data.data)) if cat_mask[i]]
    cv_cat = CountVectorizer(max_features=500, stop_words='english')
    X_cat = cv_cat.fit_transform(cat_text)
    word_freq = np.asarray(X_cat.sum(axis=0)).ravel()
    top_idx = word_freq.argsort()[::-1][:10]
    top_words = [cv_cat.get_feature_names_out()[i] for i in top_idx]
    print(f'  {cat_name:<25}: {top_words}')


## 9.2 TF-IDF Vectorizer

TF-IDF memberikan bobot yang lebih baik dibanding raw counts, mengurangi pengaruh kata-kata yang terlalu umum.


In [ ]:
# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=20000, min_df=2, stop_words='english', sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_data.data)
X_test_tfidf = tfidf.transform(test_data.data)

# Compare BoW vs TF-IDF with Naive Bayes
nb_bow = MultinomialNB(alpha=0.1)
nb_tfidf = MultinomialNB(alpha=0.1)
nb_bow.fit(X_train_bow, train_data.target)
nb_tfidf.fit(X_train_tfidf, train_data.target)

print(f'NaiveBayes + BoW  test accuracy:   {nb_bow.score(X_test_bow, test_data.target):.4f}')
print(f'NaiveBayes + TF-IDF test accuracy: {nb_tfidf.score(X_test_tfidf, test_data.target):.4f}')

# Show most discriminative words per class
feature_names = tfidf.get_feature_names_out()
print('\nMost discriminative words (highest log-prob) per category:')
for i, cat in enumerate(train_data.target_names):
    top_idx = np.argsort(nb_tfidf.feature_log_prob_[i])[-8:]
    top_words = feature_names[top_idx]
    print(f'  {cat:<25}: {list(top_words)}')


## 9.3 Klasifikasi Teks: Model Comparison

Beberapa model yang efektif untuk klasifikasi teks:

| Model | Kelebihan | Kekurangan |
|---|---|---|
| **MultinomialNB** | Cepat, baik untuk fitur sparse | Asumsi independensi kata (naive) |
| **ComplementNB** | Lebih baik untuk data tidak seimbang | - |
| **LinearSVC** | Akurasi tinggi, cepat | Tidak ada predict_proba |
| **Logistic Regression** | Probabilistik, interpretable | Lebih lambat |


In [ ]:
# Pipeline-based model comparison
models_text = {
    'MultinomialNB + BoW': Pipeline([
        ('vec', CountVectorizer(max_features=30000, min_df=2, stop_words='english')),
        ('clf', MultinomialNB(alpha=0.1))
    ]),
    'ComplementNB + TF-IDF': Pipeline([
        ('vec', TfidfVectorizer(max_features=30000, min_df=2, stop_words='english', sublinear_tf=True)),
        ('clf', ComplementNB(alpha=0.1))
    ]),
    'LinearSVC + TF-IDF': Pipeline([
        ('vec', TfidfVectorizer(max_features=30000, min_df=2, stop_words='english', sublinear_tf=True)),
        ('clf', LinearSVC(C=1.0, max_iter=2000))
    ]),
    'LR + TF-IDF (L2)': Pipeline([
        ('vec', TfidfVectorizer(max_features=30000, min_df=2, stop_words='english', sublinear_tf=True)),
        ('clf', LogisticRegression(C=5.0, max_iter=1000, solver='lbfgs', multi_class='multinomial'))
    ]),
    'LR + Bigrams TF-IDF': Pipeline([
        ('vec', TfidfVectorizer(max_features=30000, min_df=2, stop_words='english',
                                ngram_range=(1,2), sublinear_tf=True)),
        ('clf', LogisticRegression(C=5.0, max_iter=1000, solver='lbfgs', multi_class='multinomial'))
    ]),
}

all_data = train_data.data + test_data.data
all_target = np.concatenate([train_data.target, test_data.target])

print('=== Text Classification Model Comparison ===')
print(f'{"Model":<30} {"Test Acc":>10} {"5-CV Acc":>10}')
print('-' * 52)

for name, pipe in models_text.items():
    pipe.fit(train_data.data, train_data.target)
    test_acc = pipe.score(test_data.data, test_data.target)
    cv_acc = cross_val_score(pipe, all_data, all_target, cv=5).mean()
    print(f'{name:<30} {test_acc:>10.4f} {cv_acc:>10.4f}')


In [ ]:
# Confusion matrix for best model
best_text_pipe = models_text['LinearSVC + TF-IDF']
best_text_pipe.fit(train_data.data, train_data.target)
y_pred_text = best_text_pipe.predict(test_data.data)

print(classification_report(test_data.target, y_pred_text,
    target_names=[c.split('.')[-1] for c in train_data.target_names]))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    test_data.target, y_pred_text,
    display_labels=[c.split('.')[-1] for c in train_data.target_names],
    cmap='Blues', ax=ax
)
ax.set_title('LinearSVC + TF-IDF - Confusion Matrix\n20 Newsgroups (5 categories)')
plt.tight_layout()
plt.show()


## Ringkasan: Key Takeaways Chapter 9

| Konsep | Poin Kunci |
|---|---|
| **CountVectorizer** | Sederhana; sangat berguna sebagai baseline; output sparse matrix |
| **TF-IDF** | Lebih baik dari raw counts; mengurangi bobot kata umum; sublinear_tf sering membantu |
| **N-grams** | Bigrams (1,2) sering meningkatkan performa; trigrams biasanya tidak signifikan |
| **Naive Bayes** | Sangat cepat; surprisingly good untuk text; asumsi independensi kata |
| **LinearSVC** | Sering state-of-the-art untuk teks; cepat meski dimensi sangat tinggi |
| **Preprocessing** | `stop_words`, `min_df`, `max_df` penting untuk mengurangi noise |

### Pipeline Text Classification
```
raw text → Tokenization → CountVectorizer/TF-IDF → Classifier → label
```
